In [ ]:
import jax
jax.config.update('jax_enable_x64', True)  # Configure precision before other JAX imports.
import jax.numpy as jnp
import jax_md
import numpy as np

from ase.io import read
from jax_md import units
from typing import Dict

from so3lr import to_jax_md
from so3lr import So3lrPotential

import time

In [ ]:
# Some helper functions used throughout the example notebook.

# Default nose hoover chain parameters.
def default_nhc_kwargs(
    tau: jnp.float32, 
    overrides: Dict
) -> Dict:
    
    default_kwargs = {
        'chain_length': 3, 
        'chain_steps': 2, 
        'sy_steps': 3,
        'tau': tau
    }
    
    if overrides is None:
        return default_kwargs
  
    return {
      k: overrides.get(k, default_kwargs[k]) for k in default_kwargs
    }

# Read the Molecule

Start by reading a molecular structure in any format supported by ASE, such as an XYZ file.

In [ ]:
# Read some molecular structure. Here we read the DHA structure from the test directory.
path_to_xyz = '../tests/test_data/dha.xyz'
atoms = read(path_to_xyz, index=-1)

positions_init = jnp.array(atoms.get_positions())
species = jnp.array(atoms.get_atomic_numbers())
masses = jnp.array(atoms.get_masses())

# Prepare JAX-MD Energy Function

JAX-MD is built around an energy function and neighbor lists. SO3LR uses separate short- and long-range neighbor lists. The repository's `to_jax_md` convenience interface constructs the energy function and both neighbor-list functions from a `So3lrPotential`, a displacement function, and the atomic species. Here DHA is simulated in vacuum without periodic boundary conditions, so `jax_md.space.free` supplies the displacement and shift functions and positions remain Cartesian.

In [ ]:
displacement, shift = jax_md.space.free()
box = None  # No simulation box. 

neighbor_fn, neighbor_fn_lr, energy_fn = to_jax_md(
    potential=So3lrPotential(
        model='so3lr-m-2606',
        dtype=jnp.float64  # or float32 for single precision
    ),
    displacement_or_metric=displacement,
    box_size=box,
    species=species,
    capacity_multiplier=1.25,
    buffer_size_multiplier_sr=1.25,
    buffer_size_multiplier_lr=1.25,
    minimum_cell_size_multiplier_sr=1.0,
    disable_cell_list=True,  # Cell list partitioning can only be applied if there is a simulation box.
    fractional_coordinates=False
)

# Energy function.
energy_fn = jax.jit(energy_fn)
force_fn = jax.jit(jax_md.quantity.force(energy_fn))

# Initialize the short and long-range neighbor lists.
nbrs = neighbor_fn.allocate(
    positions_init,
    box=box
)
nbrs_lr = neighbor_fn_lr.allocate(
    positions_init,
    box=box
)

In [ ]:
# Check that the energy function is working.
energy_fn(
    positions_init, 
    neighbor=nbrs.idx, 
    neighbor_lr=nbrs_lr.idx, 
    box=box
)

# Structure Relaxation

In [ ]:
min_cycles = 5
min_steps = 10

fire_init, fire_apply = jax_md.minimize.fire_descent(
    energy_fn, 
    shift, 
    dt_start = 0.05, 
    dt_max = 0.1, 
    n_min = 2
)

fire_apply = jax.jit(fire_apply)
fire_state = fire_init(
    positions_init, 
    box=box, 
    neighbor=nbrs.idx,
    neighbor_lr=nbrs_lr.idx
)

@jax.jit
def step_fire_fn(i, fire_state):
    
    fire_state, nbrs, nbrs_lr = fire_state
    
    fire_state = fire_apply(
        fire_state, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    nbrs = nbrs.update(
        fire_state.position,
        neighbor=nbrs.idx
    )
    
    nbrs_lr = nbrs_lr.update(
        fire_state.position,
        neighbor_lr=nbrs_lr.idx
    )
    
    return fire_state, nbrs, nbrs_lr

print('Step\tE\tFmax')
print('----------------------------------------')
for i in range(min_cycles):
    # A chunk run with an overflowed neighbor list is invalid: reallocate both
    # lists at the last good positions and run the chunk again.
    while True:
        new_fire_state, new_nbrs, new_nbrs_lr = jax.lax.fori_loop(
            0, min_steps, step_fire_fn, (fire_state, nbrs, nbrs_lr)
        )
        if not (new_nbrs.did_buffer_overflow or new_nbrs_lr.did_buffer_overflow):
            break
        print('Neighbor list overflowed, reallocating.')
        nbrs = neighbor_fn.allocate(fire_state.position, box=box)
        nbrs_lr = neighbor_fn_lr.allocate(fire_state.position, box=box)
    fire_state, nbrs, nbrs_lr = new_fire_state, new_nbrs, new_nbrs_lr
    
    E = energy_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )

    F = force_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    print('{}\t{:.2f}\t{:.2f}'.format(i, E, np.abs(F).max()))
    

In [ ]:
# Print the delta between original and optimized positions for the first three atoms.
(positions_init - fire_state.position)[:3]

# Nose-Hoover Chain NVT Simulation

In [ ]:
# Simulation parameters

timestep = 0.0005  # Time step in ps
nvt_cycles = 5  # Number of NVT output cycles.
nvt_steps = 50  # NVT steps per cycle; total steps are nvt_cycles * nvt_steps.

T_init_kelvin = None  # Initial temperature in K; defaults to target.
T_nvt_kelvin = 300  # Target temperature in K.

chain = 3  # Number of chains in the Nose-Hoover chain.
chain_steps = 2  # Number of steps per chain.
sy_steps = 3
thermo = 100  # Thermo value in the Nose-Hoover chain. 

# Set the temperature at initialization.
if T_init_kelvin is None:
    T_init_kelvin = T_nvt_kelvin

# Dictionary with the NHC settings.
new_nhc_kwargs = {
    'chain_length': chain, 
    'chain_steps': chain_steps, 
    'sy_steps': sy_steps
}

# Convert to metal unit system.
unit = units.metal_unit_system()

timestep = timestep * unit['time']
T_init = T_init_kelvin * unit['temperature']
T_nvt = T_nvt_kelvin * unit['temperature']

rng_key = jax.random.PRNGKey(0)

In [ ]:
# Choose the Nose-Hoover thermostat.
init_fn, apply_fn = jax_md.simulate.nvt_nose_hoover(
    energy_fn, 
    shift, 
    dt=timestep, 
    kT=T_init,
    box=box,
    thermostat_kwargs=default_nhc_kwargs(thermo * timestep, new_nhc_kwargs)
)

apply_fn = jax.jit(apply_fn)
init_fn = jax.jit(init_fn)

# Initialize the state from the relaxed positions and neighbor lists.
state = init_fn(
    rng_key, 
    fire_state.position, 
    box=box, 
    neighbor=nbrs.idx, 
    neighbor_lr=nbrs_lr.idx,
    kT=T_init,
    mass=masses
)

@jax.jit
def step_nvt_fn(i, state):
    state, nbrs, nbrs_lr, box, temp_i = state
    
    state = apply_fn(
        state, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        kT=temp_i,
        box=box
    )
    
    nbrs = nbrs.update(
        state.position, 
        neighbor=nbrs.idx, 
        box = box
    )
    
    nbrs_lr = nbrs_lr.update(
        state.position, 
        neighbor_lr=nbrs_lr.idx, 
        box = box
    )
    
    return state, nbrs, nbrs_lr, box, temp_i

# Track total time and step times averaged over cycle.
total_time = time.time()

positions_md = []
temperatures_md = []

print('Step\tKE\tPE\tTotal Energy\tTemperature\tH\ttime/steps')
print('-----------------------------------------------------------------------------------')
for i in range(nvt_cycles):
    
    temp_i = T_nvt

    old_time = time.time()
    
    # Do `nvt_steps` NVT steps. A chunk run with an overflowed neighbor list is
    # invalid: reallocate both lists at the last good state and run it again.
    while True:
        new_state, new_nbrs, new_nbrs_lr, new_box, temp_i = jax.block_until_ready(
            jax.lax.fori_loop(
                0,
                nvt_steps,
                step_nvt_fn,
                (state, nbrs, nbrs_lr, box, temp_i)  # carry state is tuple
            )
        )
        if not (new_nbrs.did_buffer_overflow or new_nbrs_lr.did_buffer_overflow):
            break
        print('Neighbor list overflowed, reallocating and repeating the cycle.')
        nbrs = neighbor_fn.allocate(state.position, box=box)
        nbrs_lr = neighbor_fn_lr.allocate(state.position, box=box)
    state, nbrs, nbrs_lr, box = new_state, new_nbrs, new_nbrs_lr, new_box

    new_time = time.time()

    # Calculate some quantities for printing
    KE = jax_md.quantity.kinetic_energy(
        momentum=state.momentum,
        mass=state.mass
    )
    
    PE = energy_fn(
        state.position,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    T = jax_md.quantity.temperature(
        momentum=state.momentum,
        mass = state.mass
    ) / unit['temperature']
    
    H = jax_md.simulate.nvt_nose_hoover_invariant(
        energy_fn, 
        state, 
        kT=temp_i,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )

    positions_md.append(np.array(state.position))
    temperatures_md.append(float(T))
    
    print(f'{(i+1)*nvt_steps}\t{KE:.2f}\t{PE:.2f}\t{KE+PE:.3f}\t{T:.1f}\t{H:.3f}\t{(new_time - old_time) / nvt_steps:.4f}')

assert np.isfinite(np.asarray(positions_md)).all()
assert np.isfinite(np.asarray(temperatures_md)).all()
assert 0.1 * T_nvt_kelvin < np.median(temperatures_md) < 10 * T_nvt_kelvin

print('Total_time: ', time.time()-total_time)

# Clear all caches
jax.clear_caches()

# Visualization and IO

For this short example, positions are collected in a Python list, converted to ASE frames, and written once as EXTXYZ. For larger production runs, use the buffered HDF5 approach in `production_run.ipynb` to avoid making trajectory I/O the simulation bottleneck.

In [ ]:
# If you want to do visualization in the notebook, install nglview by doing `pip install nglview` in your virtualenv
import nglview as nv
from ase import Atoms

atoms_traj = []
for positions in positions_md:
    atoms_traj.append(
        Atoms(numbers=np.array(species), positions=positions),
    )

nv.show_asetraj(atoms_traj)

In [ ]:
# Save the frames to xyz.
from ase.io import write

trajectory_path = 'nvt_md_trajectory.xyz'
write(trajectory_path, atoms_traj, format='extxyz')
reloaded_traj = read(trajectory_path, index=':')
assert len(reloaded_traj) == nvt_cycles
assert np.isfinite(
    np.asarray([frame.positions for frame in reloaded_traj])
).all()